# SQL Database Design and Validation

This notebook validates the finalized research database schema and exports validation and report query outputs. The database stores the full pipeline audit trail from raw prices through final strategy performance.

## Research purpose

The database is used for reproducibility and auditability. Each major artifact should be traceable: raw prices, cleaned prices, returns, triplets, hedge ratios, residuals, diagnostics, events, labels, features, model runs, predictions, trades, positions, PnL, and performance summaries.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.database import (
    connect_database,
    initialize_database,
    table_row_counts,
    run_validation_queries,
    export_named_query_results,
)

DB_PATH = ROOT / "data" / "processed" / "phase16_validation_sample.db"
SCHEMA_PATH = ROOT / "sql" / "schema.sql"
VALIDATION_SQL = ROOT / "sql" / "validation_queries.sql"
REPORT_SQL = ROOT / "sql" / "report_queries.sql"
OUTPUT_DIR = ROOT / "data" / "processed"

## Initialize schema

Running this cell creates the database tables and indexes if they do not already exist.

In [ ]:
initialize_database(DB_PATH, SCHEMA_PATH)
DB_PATH

## Row counts by required table

In [ ]:
with connect_database(DB_PATH) as conn:
    row_counts = table_row_counts(conn)

row_counts.to_csv(OUTPUT_DIR / "phase16_row_counts_by_table.csv", index=False)
row_counts

## Validation query summary

Validation queries should return no issue rows. The row-count query is treated separately because it is expected to return table counts.

In [ ]:
with connect_database(DB_PATH) as conn:
    validation_summary = run_validation_queries(conn, VALIDATION_SQL)

validation_summary.to_csv(OUTPUT_DIR / "phase16_validation_query_results.csv", index=False)
validation_summary

## Export validation and report outputs

In [ ]:
with connect_database(DB_PATH) as conn:
    validation_index = export_named_query_results(conn, VALIDATION_SQL, OUTPUT_DIR / "phase16_validation_outputs", "validation")
    report_index = export_named_query_results(conn, REPORT_SQL, OUTPUT_DIR / "phase16_report_outputs", "report")

validation_index.to_csv(OUTPUT_DIR / "phase16_validation_output_index.csv", index=False)
report_index.to_csv(OUTPUT_DIR / "phase16_report_query_results.csv", index=False)
report_index

## Interpretation

A passing database validation does not mean the strategy works. It means the stored artifacts are internally consistent enough to support reproducible research, diagnostics, and later audit review.